In [2]:
import sqlite3

def init_sql_db(db_path="evaluator.db"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Create evaluation runs table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS evaluation_runs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        batch_id TEXT,
        question TEXT NOT NULL,
        response TEXT NOT NULL,
        reference TEXT,
        relevance_score REAL,
        accuracy_score REAL,
        hallucination_score REAL,
        completeness_score REAL,
        final_verdict TEXT,
        reasoning_summary TEXT
    )
    """)

    conn.commit()
    conn.close()
    print("SQLite database and tables initialized successfully.")

init_sql_db()

SQLite database and tables initialized successfully.


In [4]:
import sqlite3

def seed_evaluation_data(db_path="evaluator.db"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    sample_runs = [
        (
            "batch_001",
            "What is the capital of France?",
            "The capital of France is Paris, known for its art, gastronomy, and culture.",
            "Paris is the capital and most populous city of France.",
            1.0, 1.0, 0.0, 1.0,
            "Pass",
            "Response is accurate, fully relevant, completely answers the prompt, and contains zero hallucinations."
        ),
        (
            "batch_001",
            "Explain quantum computing in simple terms.",
            "Quantum computing uses qubits that can exist in multiple states simultaneously due to superposition, allowing them to solve complex problems faster than classical computers.",
            "Quantum computers leverage quantum mechanics phenomena such as superposition and entanglement to perform calculations.",
            0.9, 0.85, 0.1, 0.9,
            "Pass",
            "Clear and accurate explanation with minor simplification. Fully addresses the user's prompt."
        ),
        (
            "batch_002",
            "Who invented the telephone?",
            "The telephone was invented by Thomas Edison in 1890.",
            "Alexander Graham Bell was awarded the first US patent for the telephone in 1876.",
            0.8, 0.1, 0.9, 0.5,
            "Fail",
            "Major hallucination detected: Thomas Edison did not invent the telephone, and the year is incorrect according to the reference."
        )
    ]

    cursor.executemany("""
    INSERT INTO evaluation_runs (
        batch_id, question, response, reference, 
        relevance_score, accuracy_score, hallucination_score, completeness_score, 
        final_verdict, reasoning_summary
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, sample_runs)

    conn.commit()
    conn.close()
    print("Sample evaluation data inserted successfully!")

seed_evaluation_data()

Sample evaluation data inserted successfully!


In [5]:
import sqlite3
import pandas as pd

# Connect to the local SQLite database
conn = sqlite3.connect("evaluator.db")

# Execute the query and load it directly into a Pandas DataFrame
query = """
SELECT id, timestamp, batch_id, final_verdict, relevance_score, accuracy_score 
FROM evaluation_runs;
"""

df = pd.read_sql_query(query, conn)

# Close connection
conn.close()

# Display the formatted DataFrame
df

,id,timestamp,batch_id,final_verdict,relevance_score,accuracy_score
0,1,2026-10-12 19:04:34,batch_001,Pass,1.0,1.00
1,2,2026-10-12 19:04:34,batch_001,Pass,0.9,0.85
2,3,2026-10-12 19:04:34,batch_002,Fail,0.8,0.10
3,4,2026-10-12 19:04:39,batch_001,Pass,1.0,1.00
4,5,2026-10-12 19:04:39,batch_001,Pass,0.9,0.85
5,6,2026-10-12 19:04:39,batch_002,Fail,0.8,0.10


In [6]:
import sqlite3
import pandas as pd

# Connect to the local database
conn = sqlite3.connect("evaluator.db")

# Prepare the aggregate SQL query
query = """
SELECT 
    AVG(relevance_score) AS avg_relevance,
    AVG(accuracy_score) AS avg_accuracy,
    AVG(hallucination_score) AS avg_hallucination,
    AVG(completeness_score) AS avg_completeness
FROM evaluation_runs;
"""

# Read SQL results into a pandas DataFrame
df_avg = pd.read_sql_query(query, conn)

# Close the connection


# Display the results
df_avg

,avg_relevance,avg_accuracy,avg_hallucination,avg_completeness
0,0.9,0.65,0.333333,0.8


In [ ]:
import sqlite3

def run_queries(db_path="evaluator.db"):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    print("-- 1. Viewing Failed Evaluations ---")
    cursor.execute("""
        SELECT question, response, reasoning_summary 
        FROM evaluation_runs 
        WHERE final_verdict = 'Fail'
    """)
    rows = cursor.fetchall()
    for row in rows:
        print(f"Question: {row[0]}")
        print(f"Response: {row[1]}")
        print(f"Reasoning: {row[2]}\n")

    print("--- 2. Average Scores Across All Runs ---")
    cursor.execute("""
        SELECT 
            AVG(relevance_score),
            AVG(accuracy_score),
            AVG(hallucination_score),
            AVG(completeness_score)
        FROM evaluation_runs
    """)
    avgs = cursor.fetchone()
    print(f"Avg Relevance: {avgs[0]:.2f}")
    print(f"Avg Accuracy: {avgs[1]:.2f}")
    print(f"Avg Hallucination: {avgs[2]:.2f}")
    print(f"Avg Completeness: {avgs[3]:.2f}")

    conn.close()

run_queries()

--- 1. Viewing Failed Evaluations ---
Question: Who invented the telephone?
Response: The telephone was invented by Thomas Edison in 1890.
Reasoning: Major hallucination detected: Thomas Edison did not invent the telephone, and the year is incorrect according to the reference.

Question: Who invented the telephone?
Response: The telephone was invented by Thomas Edison in 1890.
Reasoning: Major hallucination detected: Thomas Edison did not invent the telephone, and the year is incorrect according to the reference.

--- 2. Average Scores Across All Runs ---
Avg Relevance: 0.90
Avg Accuracy: 0.65
Avg Hallucination: 0.33
Avg Completeness: 0.80
